# FDS Challenge: My Notebook

### 0. Debug Settings

In [ ]:
debug = True
output_path = "first_battle.text"


def print_first_battle(data):

    if not data:
        return
    
    first_battle = data[0]

    # To keep the output clean, we can create a copy and truncate the timeline
    battle_for_display = first_battle.copy()
    battle_for_display["battle_timeline"] = battle_for_display.get("battle_timeline", [])[:30]

    # Use json.dumps for pretty-printing the dictionary
    with open(output_path, "w") as f:
        f.write(json.dumps(first_battle["battle_timeline"], indent=4))

    print("-" * 50)
    print("FIRST BATTLE TIMELINE WRITTEN TO " + output_path)
    print("-" * 50)


### 1. Loading the Data


In [ ]:
import json
import pandas as pd
from pathlib import Path

train_file_path = "./dataset/train.jsonl"
test_file_path = "./dataset/test.jsonl"
sample_submission_path = "./dataset/sample_submission.csv"

# TEST
try:
    test_data = []
    with open(test_file_path, "r") as f:
        for line in f:
            test_data.append(json.loads(line))

except FileNotFoundError:
    print(f"ERROR: Could not find the training file at '{train_file_path}'.")
    print("Please make sure you have added the competition data to this notebook.")

# TRAIN
print(f"Loading data from '{train_file_path}'...")
train_data = []
try:
    with open(train_file_path, "r") as f:
        for line in f:
            train_data.append(json.loads(line))
    if debug:
        print_first_battle(train_data)

except FileNotFoundError:
    print(f"ERROR: Could not find the training file at '{train_file_path}'.")
    print("Please make sure you have added the competition data to this notebook.")

### 2. Extract Features (both Train and Test Features)

Now we are only using the number of dead pokemons (identified by status = "fnt") as feature. The assumption is that "more dead pokemons => less likelihood to win"


In [ ]:
def extract_features(train_data):
    features_list:list[dict] = []
    for battle in train_data:
        features = {}
        for turn in battle["battle_timeline"]:
            pokemon_p1 = turn["p1_pokemon_state"]
            pokemon_p2 = turn["p2_pokemon_state"]

            # Estraiamo le feature riguardanti i pokemon esausti
            if pokemon_p1["status"] == "fnt":
                features["p1_fnt"] = features.get("p1_fnt", 0) + 1
            if pokemon_p2["status"] == "fnt":
                features["p2_fnt"] = features.get("p2_fnt", 0) + 1

        # Valorizzazione dei campi presenti dentro il train dataset (non presenti nei test)        
        if "player_won" in battle:
            features["player_won"] = battle["player_won"]
        if "battle_id" in battle:
            features["battle_id"] = battle["battle_id"]

        features_list.append(features)

    return pd.DataFrame(features_list).fillna(0)


train_df = extract_features(train_data)
display(train_df.head())

test_df = extract_features(test_data)

# Perfomance Metrics

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


def compute_metrics(model, X_val, y_val):
    y_pred = model.predict(X_val)
    y_score = model.predict_proba(X_val)[:,1]

    acc = accuracy_score(y_val, y_pred)

    prec = precision_score(y_val, y_pred)

    rec = recall_score(y_val, y_pred)

    f1 = f1_score(y_val, y_pred)

    roc = roc_auc_score(y_val, y_score)

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "roc_auc": roc,
    }

### 3. Training a Baseline Model

Now that we have some features, let's train a simple `LogisticRegression` model. This will give us a starting point for our predictions.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

# Define our features (X) and target (y)
features = [col for col in train_df.columns if col not in ["battle_id", "player_won"]]
X = train_df[features]
y = train_df["player_won"]

K = 5  # Number of k-fold splits
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=42)

scores=[]

print(f"Starting K-fold Regression model training. K={K}")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    # Split the Features and Target in train and validation
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Initialize and train the model
    print(f"Starting training {fold+1}/{K}...")
    model = LogisticRegression(random_state=42, max_iter=1000)  # Remember to set random state!
    model.fit(X_train, y_train)
    print(f"Training {fold+1}/{K} complete!")
    metrics = compute_metrics(model,X_val, y_val)  
    print(f"Metrics for fold {fold+1}: {metrics}")
    scores.append(metrics)
    

# Align columns - crucial for ensuring test set has same features as train set
X_test = test_df[features]

### 4. Creating the Submission File

The competition requires a `.csv` file with two columns: `battle_id` and `player_won`. Let's use our trained model to make predictions on the test set and format them correctly.

In [ ]:
# Make predictions on the test data
print("Generating predictions on the test set...")
test_predictions = model.predict(X_test)

# Use the provided sample file as the submission template.
sample_submission = pd.read_csv(sample_submission_path)
expected_columns = ["battle_id", "player_won"]
if list(sample_submission.columns) != expected_columns:
    raise ValueError(f"Expected submission columns {expected_columns}, got {list(sample_submission.columns)}")

# Match predictions to the template by battle_id, preserving its row order.
predictions_by_id = pd.Series(test_predictions, index=test_df["battle_id"])
submission_df = sample_submission.copy()
submission_df["player_won"] = submission_df["battle_id"].map(predictions_by_id)
if submission_df["player_won"].isna().any():
    raise ValueError("Some sample submission battle_id values have no matching test prediction.")
submission_df["player_won"] = submission_df["player_won"].astype(int)

submission_df.to_csv("submission.csv", index=False)

# Once you create the submission file, you will find it in /kaggle/working
print("\n'submission.csv' file created successfully!")
print(submission_df.head())